# **Configuration**

In [22]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 24, Finished, Available, Finished, False)

In [23]:
print("Database:", spark.catalog.currentDatabase())

spark.sql("SHOW SCHEMAS").show(truncate=False)

spark.sql("SHOW TABLES IN dbo").show(100, truncate=False)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 25, Finished, Available, Finished, False)

Database: chimcobldhq2al35dhim6rrd5l1mgtbidoml0sj5chkm6t39dtn2qgbec5m7it39cdpiar38btq6ar35cdnmqnrjd5m7cpbi4li64ro
+--------------------------------------------------------+
|namespace                                               |
+--------------------------------------------------------+
|Telecom-Churn-Prediction-Analytics.lh_telecom_silver.dbo|
+--------------------------------------------------------+

+----------------------------------------------------------+-------------------------------+-----------+
|namespace                                                 |tableName                      |isTemporary|
+----------------------------------------------------------+-------------------------------+-----------+
|`Telecom-Churn-Prediction-Analytics`.lh_telecom_silver.dbo|quarantine_billing_churn       |false      |
|`Telecom-Churn-Prediction-Analytics`.lh_telecom_silver.dbo|quarantine_crm_customer        |false      |
|`Telecom-Churn-Prediction-Analytics`.lh_telecom_silver.dbo|quara

# **Read and Validate**

In [24]:
SILVER_SCHEMA = "dbo"
GOLD_LAKEHOUSE = "lh_telecom_gold"

CRM_TABLE = f"{SILVER_SCHEMA}.silver_crm_customer"
BILLING_TABLE = f"{SILVER_SCHEMA}.silver_billing_churn"
SERVICE_TABLE = f"{SILVER_SCHEMA}.silver_service_provisioning"
CUSTOMER_ID = "customer_id"

crm_df = spark.table(CRM_TABLE)
billing_df = spark.table(BILLING_TABLE)
service_df = spark.table(SERVICE_TABLE)

print("CRM rows:", crm_df.count())
print("Billing rows:", billing_df.count())
print("Service rows:", service_df.count())

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 26, Finished, Available, Finished, False)

CRM rows: 6857
Billing rows: 6857
Service rows: 6857


In [25]:
for table_name, df in {
    "CRM": crm_df,
    "Billing": billing_df,
    "Service": service_df
}.items():

    if CUSTOMER_ID not in df.columns:
        raise ValueError(
            f"{CUSTOMER_ID} does not exist in {table_name}. "
            f"Available columns: {df.columns}"
        )
    else:
        print(f"{CUSTOMER_ID} exits in {table_name}")
        

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 27, Finished, Available, Finished, False)

customer_id exits in CRM
customer_id exits in Billing
customer_id exits in Service


In [26]:
print("CRM columns")
print(crm_df.columns)

print("\nBilling columns")
print(billing_df.columns)

print("\nService columns")
print(service_df.columns)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 28, Finished, Available, Finished, False)

CRM columns
['customer_id', 'gender', 'senior_citizen', 'partner', 'dependent', 'contract', 'paperless_billing', 'payment_method', 'country', 'state', 'city', 'zip_code', 'latitude', 'longitude']

Billing columns
['customer_id', 'monthly_charges', 'total_charges', 'churn_label', 'churn_reason', 'churn_value']

Service columns
['customer_id', 'tenure_months', 'phone_service', 'multiple_lines', 'internet_service', 'online_security', 'online_backup', 'device_protection', 'tech_support', 'streaming_tv', 'streaming_movies']


In [27]:
def validate_customer_grain(df, table_name, customer_id_col="customer_id"):
    total_rows = df.count()

    distinct_customers = (
        df.select(customer_id_col)
        .distinct()
        .count()
    )

    null_ids = (
        df.filter(F.col(customer_id_col).isNull())
        .count()
    )

    blank_ids = (
        df.filter(F.trim(F.col(customer_id_col)) == "")
        .count()
    )

    duplicate_ids = (
        df.groupBy(customer_id_col)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(f"{table_name}")
    print(f"Total rows: {total_rows:,}")
    print(f"Distinct customers: {distinct_customers:,}")
    print(f"Null customer IDs: {null_ids:,}")
    print(f"Blank customer IDs: {blank_ids:,}")
    print(f"Duplicated customer IDs: {duplicate_ids:,}")
    print("-" * 60)

    if null_ids > 0 or blank_ids > 0 or duplicate_ids > 0:
        raise ValueError(
            f"{table_name} does not have a valid one-row-per-customer grain."
        )

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 29, Finished, Available, Finished, False)

In [28]:
validate_customer_grain(crm_df, "CRM")
validate_customer_grain(billing_df, "Billing")
validate_customer_grain(service_df, "Service")

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 30, Finished, Available, Finished, False)

CRM
Total rows: 6,857
Distinct customers: 6,857
Null customer IDs: 0
Blank customer IDs: 0
Duplicated customer IDs: 0
------------------------------------------------------------
Billing
Total rows: 6,857
Distinct customers: 6,857
Null customer IDs: 0
Blank customer IDs: 0
Duplicated customer IDs: 0
------------------------------------------------------------
Service
Total rows: 6,857
Distinct customers: 6,857
Null customer IDs: 0
Blank customer IDs: 0
Duplicated customer IDs: 0
------------------------------------------------------------


In [29]:
def count_unmatched(left_df, right_df, left_name, right_name):
    unmatched_df = (
        left_df.select(CUSTOMER_ID)
        .join(
            right_df.select(CUSTOMER_ID),
            on=CUSTOMER_ID,
            how="left_anti"
        )
    )

    unmatched_count = unmatched_df.count()

    print(
        f"{left_name} customers missing from "
        f"{right_name}: {unmatched_count:,}"
    )

    return unmatched_df

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 31, Finished, Available, Finished, False)

In [30]:
crm_missing_billing = count_unmatched(
    crm_df,
    billing_df,
    "CRM",
    "Billing"
)

billing_missing_crm = count_unmatched(
    billing_df,
    crm_df,
    "Billing",
    "CRM"
)

crm_missing_service = count_unmatched(
    crm_df,
    service_df,
    "CRM",
    "Service"
)

service_missing_crm = count_unmatched(
    service_df,
    crm_df,
    "Service",
    "CRM"
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 32, Finished, Available, Finished, False)

CRM customers missing from Billing: 0
Billing customers missing from CRM: 0
CRM customers missing from Service: 0
Service customers missing from CRM: 0


# **Creating Customer Status Column**

In [31]:
billing_df = (
    billing_df
    .withColumn(
        "customer_status",
        F.when(
            F.col("churn_value") == 1,
            "Churned"
        )
        .when(
            F.col("churn_value") == 0,
            "Stayed"
        )
        .otherwise("Unknown")
    )
)

display(billing_df.limit(5))

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6d630c04-5ade-4ac6-8f9b-9b7eb8b5e79e)

# **Creating a combined customer-level staging DataFrame**

In [32]:
crm_columns = set(crm_df.columns)
billing_columns = set(billing_df.columns)
service_columns = set(service_df.columns)

crm_billing_overlap = crm_columns.intersection(billing_columns)
crm_service_overlap = crm_columns.intersection(service_columns)
billing_service_overlap = billing_columns.intersection(service_columns)

print("CRM and Billing shared columns:")
print(crm_billing_overlap)

print("\nCRM and Service shared columns:")
print(crm_service_overlap)

print("\nBilling and Service shared columns:")
print(billing_service_overlap)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 34, Finished, Available, Finished, False)

CRM and Billing shared columns:
{'customer_id'}

CRM and Service shared columns:
{'customer_id'}

Billing and Service shared columns:
{'customer_id'}


In [33]:
def prefix_columns(df, prefix, join_column="customer_id"):
    selected_columns = []

    for column_name in df.columns:
        if column_name == join_column:
            selected_columns.append(F.col(column_name))
        else:
            selected_columns.append(
                F.col(column_name).alias(f"{prefix}_{column_name}")
            )

    return df.select(*selected_columns)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 35, Finished, Available, Finished, False)

In [34]:
crm_prefixed_df = prefix_columns(crm_df, "crm")
billing_prefixed_df = prefix_columns(billing_df, "billing")
service_prefixed_df = prefix_columns(service_df, "service")

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 36, Finished, Available, Finished, False)

In [35]:
gold_staging_df = (
    crm_prefixed_df
    .join(
        billing_prefixed_df,
        on=CUSTOMER_ID,
        how="inner"
    )
    .join(
        service_prefixed_df,
        on=CUSTOMER_ID,
        how="inner"
    )
)

print("Gold staging rows:", gold_staging_df.count())
print("Gold staging columns:", len(gold_staging_df.columns))

display(gold_staging_df.limit(10))

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 37, Finished, Available, Finished, False)

Gold staging rows: 6857
Gold staging columns: 30


SynapseWidget(Synapse.DataFrame, b777924a-b100-4246-86fe-e6d9115bfe40)

In [36]:
validate_customer_grain(
    gold_staging_df,
    "Gold staging",
    CUSTOMER_ID
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 38, Finished, Available, Finished, False)

Gold staging
Total rows: 6,857
Distinct customers: 6,857
Null customer IDs: 0
Blank customer IDs: 0
Duplicated customer IDs: 0
------------------------------------------------------------


In [37]:
gold_staging_df.printSchema()

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 39, Finished, Available, Finished, False)

root
 |-- customer_id: string (nullable = true)
 |-- crm_gender: string (nullable = true)
 |-- crm_senior_citizen: string (nullable = true)
 |-- crm_partner: string (nullable = true)
 |-- crm_dependent: string (nullable = true)
 |-- crm_contract: string (nullable = true)
 |-- crm_paperless_billing: string (nullable = true)
 |-- crm_payment_method: string (nullable = true)
 |-- crm_country: string (nullable = true)
 |-- crm_state: string (nullable = true)
 |-- crm_city: string (nullable = true)
 |-- crm_zip_code: string (nullable = true)
 |-- crm_latitude: double (nullable = true)
 |-- crm_longitude: double (nullable = true)
 |-- billing_monthly_charges: double (nullable = true)
 |-- billing_total_charges: double (nullable = true)
 |-- billing_churn_label: string (nullable = true)
 |-- billing_churn_reason: string (nullable = true)
 |-- billing_churn_value: integer (nullable = true)
 |-- billing_customer_status: string (nullable = false)
 |-- service_tenure_months: integer (nullable = t

In [38]:
for column_name in gold_staging_df.columns:
    print(column_name)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 40, Finished, Available, Finished, False)

customer_id
crm_gender
crm_senior_citizen
crm_partner
crm_dependent
crm_contract
crm_paperless_billing
crm_payment_method
crm_country
crm_state
crm_city
crm_zip_code
crm_latitude
crm_longitude
billing_monthly_charges
billing_total_charges
billing_churn_label
billing_churn_reason
billing_churn_value
billing_customer_status
service_tenure_months
service_phone_service
service_multiple_lines
service_internet_service
service_online_security
service_online_backup
service_device_protection
service_tech_support
service_streaming_tv
service_streaming_movies


# **Creating Dim Tables**

# **Dimension Keys**

In [50]:
customer_key_expr = F.xxhash64(
    F.col("customer_id")
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 52, Finished, Available, Finished, False)

In [51]:
geography_key_expr = F.xxhash64(
    F.coalesce(F.col("crm_country"), F.lit("UNKNOWN")),
    F.coalesce(F.col("crm_state"), F.lit("UNKNOWN")),
    F.coalesce(F.col("crm_city"), F.lit("UNKNOWN")),
    F.coalesce(F.col("crm_zip_code").cast("string"), F.lit("UNKNOWN")),
    F.coalesce(F.col("crm_latitude").cast("string"), F.lit("UNKNOWN")),
    F.coalesce(F.col("crm_longitude").cast("string"), F.lit("UNKNOWN"))
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 53, Finished, Available, Finished, False)

In [52]:
contract_key_expr = F.xxhash64(
    F.coalesce(F.col("crm_contract"), F.lit("UNKNOWN")),
    F.coalesce(F.col("crm_paperless_billing"), F.lit("UNKNOWN")),
    F.coalesce(F.col("crm_payment_method"), F.lit("UNKNOWN"))
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 54, Finished, Available, Finished, False)

In [53]:
service_key_expr = F.xxhash64(
    F.coalesce(F.col("service_phone_service"), F.lit("UNKNOWN")),
    F.coalesce(F.col("service_multiple_lines"), F.lit("UNKNOWN")),
    F.coalesce(F.col("service_internet_service"), F.lit("UNKNOWN")),
    F.coalesce(F.col("service_online_security"), F.lit("UNKNOWN")),
    F.coalesce(F.col("service_online_backup"), F.lit("UNKNOWN")),
    F.coalesce(F.col("service_device_protection"), F.lit("UNKNOWN")),
    F.coalesce(F.col("service_tech_support"), F.lit("UNKNOWN")),
    F.coalesce(F.col("service_streaming_tv"), F.lit("UNKNOWN")),
    F.coalesce(F.col("service_streaming_movies"), F.lit("UNKNOWN")),
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 55, Finished, Available, Finished, False)

# **Dim customer**

In [54]:
dim_customer = (
    gold_staging_df
    .select(
        customer_key_expr.alias("customer_key"),
        F.col("customer_id"),
        F.col("crm_gender").alias("gender"),
        F.col("crm_senior_citizen").alias("senior_citizen"),
        F.col("crm_partner").alias("partner"),
        F.col("crm_dependent").alias("dependents"),
        F.col("billing_customer_status")
            .alias("customer_status")
    )
    .dropDuplicates(["customer_key"])
    .withColumn(
        "gold_created_timestamp",
        F.current_timestamp()
    )
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 56, Finished, Available, Finished, False)

In [55]:
display(dim_customer.limit(10))
dim_customer.printSchema()

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 57, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ee91ce71-8357-4002-8fd9-6d7c5ecd4553)

root
 |-- customer_key: long (nullable = false)
 |-- customer_id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- senior_citizen: string (nullable = true)
 |-- partner: string (nullable = true)
 |-- dependents: string (nullable = true)
 |-- customer_status: string (nullable = false)
 |-- gold_created_timestamp: timestamp (nullable = false)



In [56]:
print("Dimension rows:", dim_customer.count())

print(
    "Distinct customer keys:",
    dim_customer.select("customer_key").distinct().count()
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 58, Finished, Available, Finished, False)

Dimension rows: 6857
Distinct customer keys: 6857


# **Dim geography**

In [57]:
dim_geography = (
    gold_staging_df
    .select(
        geography_key_expr.alias("geography_key"),

        F.col("crm_country").alias("country"),
        F.col("crm_state").alias("state"),
        F.col("crm_city").alias("city"),
        F.col("crm_zip_code").cast("string").alias("zip_code"),
        F.col("crm_latitude").cast("double").alias("latitude"),
        F.col("crm_longitude").cast("double").alias("longitude")
    )
    .dropDuplicates(["geography_key"])
    .withColumn(
        "gold_created_timestamp",
        F.current_timestamp()
    )
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 59, Finished, Available, Finished, False)

In [58]:
display(dim_geography.limit(10))

dim_geography.printSchema()

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 60, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6885bf71-7158-4541-b0ab-0bd6f552102e)

root
 |-- geography_key: long (nullable = false)
 |-- country: string (nullable = true)
 |-- state: string (nullable = true)
 |-- city: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- gold_created_timestamp: timestamp (nullable = false)



# **Dim contract**

In [59]:
dim_contract = (
    gold_staging_df
    .select(
        contract_key_expr.alias("contract_key"),

        F.col("crm_contract")
            .alias("contract_type"),

        F.col("crm_paperless_billing")
            .alias("paperless_billing"),

        F.col("crm_payment_method")
            .alias("payment_method")
    )
    .dropDuplicates(["contract_key"])
    .withColumn(
        "gold_created_timestamp",
        F.current_timestamp()
    )
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 61, Finished, Available, Finished, False)

In [60]:
display(dim_contract)

dim_contract.printSchema()

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 62, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3ace2e99-9973-499a-8b47-7002f4a70761)

root
 |-- contract_key: long (nullable = false)
 |-- contract_type: string (nullable = true)
 |-- paperless_billing: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- gold_created_timestamp: timestamp (nullable = false)



# **Dim service**

In [61]:
dim_service = (
    gold_staging_df
    .select(
        service_key_expr.alias("service_key"),

        F.col("service_phone_service")
            .alias("phone_service"),

        F.col("service_multiple_lines")
            .alias("multiple_lines"),

        F.col("service_internet_service")
            .alias("internet_service"),

        F.col("service_online_security")
            .alias("online_security"),

        F.col("service_online_backup")
            .alias("online_backup"),

        F.col("service_device_protection")
            .alias("device_protection"),

        F.col("service_tech_support")
            .alias("tech_support"),

        F.col("service_streaming_tv")
            .alias("streaming_tv"),

        F.col("service_streaming_movies")
            .alias("streaming_movies")
    )
    .dropDuplicates(["service_key"])
    .withColumn(
        "gold_created_timestamp",
        F.current_timestamp()
    )
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 63, Finished, Available, Finished, False)

In [62]:
display(dim_service.limit(10))

dim_service.printSchema()

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 64, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1ba3315f-6105-4bd6-9bfe-0264cf059be1)

root
 |-- service_key: long (nullable = false)
 |-- phone_service: string (nullable = true)
 |-- multiple_lines: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- online_security: string (nullable = true)
 |-- online_backup: string (nullable = true)
 |-- device_protection: string (nullable = true)
 |-- tech_support: string (nullable = true)
 |-- streaming_tv: string (nullable = true)
 |-- streaming_movies: string (nullable = true)
 |-- gold_created_timestamp: timestamp (nullable = false)



# **Creatng fact-table**

In [67]:
fact_customer_churn = (
    gold_staging_df
    .select(
        # Foreign keys
        customer_key_expr.alias("customer_key"),
        geography_key_expr.alias("geography_key"),
        contract_key_expr.alias("contract_key"),
        service_key_expr.alias("service_key"),

        # Natural key
        F.col("customer_id"),

        # Measures
        F.col("service_tenure_months")
            .cast("int")
            .alias("tenure_months"),

        F.col("billing_monthly_charges")
            .cast("double")
            .alias("monthly_charges"),

        F.col("billing_total_charges")
            .cast("double")
            .alias("total_charges"),

        # Churn fields
        F.col("billing_churn_value")
            .cast("int")
            .alias("churn_flag"),

        F.col("billing_churn_label")
            .cast("string")
            .alias("churn_label"),

        F.col("billing_churn_reason")
            .cast("string")
            .alias("churn_reason"),

        F.col("billing_churn_value")
            .cast("int")
            .alias("customer_status")
    )
    .withColumn(
        "gold_created_timestamp",
        F.current_timestamp()
    )
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 69, Finished, Available, Finished, False)

In [68]:
fact_customer_churn = (
    fact_customer_churn
    .withColumn(
        "tenure_group",
        F.when(
            F.col("tenure_months").isNull(),
            "Unknown"
        )
        .when(
            F.col("tenure_months") <= 6,
            "0-6 Months"
        )
        .when(
            F.col("tenure_months") <= 12,
            "7-12 Months"
        )
        .when(
            F.col("tenure_months") <= 24,
            "13-24 Months"
        )
        .when(
            F.col("tenure_months") <= 48,
            "25-48 Months"
        )
        .otherwise("49+ Months")
    )
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 70, Finished, Available, Finished, False)

In [69]:
fact_customer_churn = (
    fact_customer_churn
    .withColumn(
        "monthly_charge_band",
        F.when(
            F.col("monthly_charges").isNull(),
            "Unknown"
        )
        .when(
            F.col("monthly_charges") < 30,
            "Low"
        )
        .when(
            F.col("monthly_charges") < 70,
            "Medium"
        )
        .otherwise("High")
    )
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 71, Finished, Available, Finished, False)

In [70]:
fact_customer_churn = fact_customer_churn.select(
    "customer_key",
    "geography_key",
    "contract_key",
    "service_key",

    "customer_id",

    "tenure_months",
    "monthly_charges",
    "total_charges",

    "churn_flag",
    "churn_label",
    "churn_reason",
    "customer_status",

    "tenure_group",
    "monthly_charge_band",

    "gold_created_timestamp"
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 72, Finished, Available, Finished, False)

In [72]:
display(fact_customer_churn)

fact_customer_churn.printSchema()

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 74, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3cfb37d9-5b33-4ea3-80d1-9d07a26212ce)

root
 |-- customer_key: long (nullable = false)
 |-- geography_key: long (nullable = false)
 |-- contract_key: long (nullable = false)
 |-- service_key: long (nullable = false)
 |-- customer_id: string (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_charges: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- churn_flag: integer (nullable = true)
 |-- churn_label: string (nullable = true)
 |-- churn_reason: string (nullable = true)
 |-- customer_status: integer (nullable = true)
 |-- tenure_group: string (nullable = false)
 |-- monthly_charge_band: string (nullable = false)
 |-- gold_created_timestamp: timestamp (nullable = false)



# **Validate Star Schema**

In [73]:
#Foreign Key validation
assert dim_customer.count() == dim_customer.select("customer_key").distinct().count()

assert dim_geography.count() == dim_geography.select("geography_key").distinct().count()

assert dim_contract.count() == dim_contract.select("contract_key").distinct().count()

assert dim_service.count() == dim_service.select("service_key").distinct().count()

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 75, Finished, Available, Finished, False)

In [83]:
foreign_keys = [
    "customer_key",
    "geography_key",
    "contract_key",
    "service_key"
]

for key in foreign_keys:
    null_count = (
        fact_customer_churn
        .filter(F.col(key).isNull())
        .count()
    )

    print(f"{key}: {null_count:,} null values")

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 85, Finished, Available, Finished, False)

customer_key: 0 null values
geography_key: 0 null values
contract_key: 0 null values
service_key: 0 null values


In [82]:
validate_customer_grain(
    fact_customer_churn,
    "Fact Customer Churn",
    CUSTOMER_ID
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 84, Finished, Available, Finished, False)

Fact Customer Churn
Total rows: 6,857
Distinct customers: 6,857
Null customer IDs: 0
Blank customer IDs: 0
Duplicated customer IDs: 0
------------------------------------------------------------


In [84]:
def validate_foreign_key(
    fact_df,
    dimension_df,
    key_column,
    dimension_name
):
    orphan_count = (
        fact_df.select(key_column)
        .distinct()
        .join(
            dimension_df.select(key_column).distinct(),
            on=key_column,
            how="left_anti"
        )
        .count()
    )

    print(
        f"Orphan {key_column} values against "
        f"{dimension_name}: {orphan_count:,}"
    )

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 86, Finished, Available, Finished, False)

In [85]:
validate_foreign_key(
    fact_customer_churn,
    dim_customer,
    "customer_key",
    "dim_customer"
)

validate_foreign_key(
    fact_customer_churn,
    dim_geography,
    "geography_key",
    "dim_geography"
)

validate_foreign_key(
    fact_customer_churn,
    dim_contract,
    "contract_key",
    "dim_contract"
)

validate_foreign_key(
    fact_customer_churn,
    dim_service,
    "service_key",
    "dim_service"
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 87, Finished, Available, Finished, False)

Orphan customer_key values against dim_customer: 0
Orphan geography_key values against dim_geography: 0
Orphan contract_key values against dim_contract: 0
Orphan service_key values against dim_service: 0


In [81]:
fact_customer_churn.select(
    F.count("*").alias("rows"),
    F.sum(F.col("monthly_charges").isNull().cast("int")).alias("null_monthly"),
    F.sum(F.col("total_charges").isNull().cast("int")).alias("null_total")
).show()

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 83, Finished, Available, Finished, False)

+----+------------+----------+
|rows|null_monthly|null_total|
+----+------------+----------+
|6857|           0|         0|
+----+------------+----------+



# **Write tables into the Gold Lakehouse**

In [86]:
def write_gold_table(df, table_name):
    full_table_name = f"{GOLD_LAKEHOUSE}.dbo.{table_name}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table_name)
    )

    print(f"Saved: {full_table_name}")

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 88, Finished, Available, Finished, False)

In [87]:
write_gold_table(
    dim_customer,
    "dim_customer"
)

write_gold_table(
    dim_geography,
    "dim_geography"
)

write_gold_table(
    dim_contract,
    "dim_contract"
)

write_gold_table(
    dim_service,
    "dim_service"
)

write_gold_table(
    fact_customer_churn,
    "fact_customer_churn"
)

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 89, Finished, Available, Finished, False)

Saved: lh_telecom_gold.dbo.dim_customer
Saved: lh_telecom_gold.dbo.dim_geography
Saved: lh_telecom_gold.dbo.dim_contract
Saved: lh_telecom_gold.dbo.dim_service
Saved: lh_telecom_gold.dbo.fact_customer_churn


In [88]:
gold_tables = [
    "dim_customer",
    "dim_geography",
    "dim_contract",
    "dim_service",
    "fact_customer_churn"
]

for table_name in gold_tables:
    full_name = f"{GOLD_LAKEHOUSE}.dbo.{table_name}"
    row_count = spark.table(full_name).count()

    print(f"{table_name:<30} {row_count:>10,} rows")

StatementMeta(, 92d447ab-078b-4866-9387-687bd4838102, 90, Finished, Available, Finished, False)

dim_customer                        6,857 rows
dim_geography                       1,652 rows
dim_contract                           24 rows
dim_service                           322 rows
fact_customer_churn                 6,857 rows
